# Race
- Runs all `NUM_RACES` races per circuit automatically; both reuse the same Qualifying grid from `grid.md`
- Distance: `ceil(100 / lap_length_km)` laps
- Weather rolls fresh per race; tyre wear, fuel load, crash risk all modelled
- Outputs `race{N}_result.md` per race, plus a single overall `ranking.md` (riders / teams / manufacturers totalled across all races)

In [38]:
import pandas as pd
import numpy as np
import math
from pathlib import Path

RAW       = Path('../data/raw')
MAX_DELTA = 3.0
NUM_RACES = 2   # races held at each circuit (both reuse the Qualifying grid)

In [39]:
entry_info    = pd.read_csv(RAW / 'entry_info.csv')
riders_rating = pd.read_csv(RAW / 'riders_rating.csv').rename(columns={
    'braking': 'rider_braking', 'cornering': 'rider_cornering'
})
bikes_rating  = pd.read_csv(RAW / 'bikes_rating.csv').rename(columns={
    'braking': 'bike_braking', 'cornering': 'bike_cornering'
})
circuits = pd.read_csv(RAW / 'circuits.csv')

df = (
    entry_info
    .merge(riders_rating, on='name', how='left')
    .merge(bikes_rating,  on=['manufacturer', 'team_status'], how='left')
)

# ── Circuit (must match Practice / Qualifying notebooks) ──────────────────
CIRCUIT_IDX = 0
circuit    = circuits.iloc[CIRCUIT_IDX]
country    = circuit['country']
report_dir = Path('../report') / country
base_time  = float(circuit['base_lap_time'])
total_laps = math.ceil(100 / circuit['lap_length_km'])

print(f"Circuit    : {circuit['circuit_name']} ({country})")
print(f"Race laps  : {total_laps}  ({circuit['lap_length_km']} km × {total_laps} = {circuit['lap_length_km']*total_laps:.1f} km)")

# ── Parse starting grid from grid.md (shared by both races) ───────────────
grid_md   = (report_dir / 'grid.md').read_text(encoding='utf-8')
grid_rows = [l for l in grid_md.split('\n') if l.startswith('| P') and not l.startswith('| P |')]
grid_order = []
for line in grid_rows:
    parts = [p.strip() for p in line.split('|')]
    grid_order.append({
        'grid_pos'   : int(parts[1][1:]),
        'bike_number': int(parts[2][1:]),
        'name'       : parts[3],
    })

# Merge on both name + bike_number to avoid duplicate columns
grid_df = (
    pd.DataFrame(grid_order)
    .sort_values('grid_pos')
    .merge(df, on=['name', 'bike_number'], how='left')
)

print(f"\nStarting grid loaded: {len(grid_df)} riders")
print(grid_df[['grid_pos', 'bike_number', 'name', 'team']].to_string(index=False))

Circuit    : Liam Henderson Circuit (Australia)
Race laps  : 23  (4.4 km × 23 = 101.2 km)

Starting grid loaded: 24 riders
 grid_pos  bike_number                  name                    team
        1           91      Mehmet Terzioğlu   Ducati Factory Racing
        2           44         Lorenzo Russo   Yamaha Factory Racing
        3           31       Niklas Hoffmann   Suzuki Factory Racing
        4           88     Sebastian Aginaza            Razor Racing
        5           76            João Sousa   Suzuki Factory Racing
        6           20            Kaito Sato Kawasaki Factory Racing
        7           90        Nathan Stewart   Ducati Factory Racing
        8           19       Matteo Esposito    Honda Factory Racing
        9           26           Javier Ruiz            Razor Racing
       10           63        Daniel Vaquero Kawasaki Factory Racing
       11           32           Dwi Gunawan           Falcon Racing
       12            7       Petros Georgiou    H

In [40]:
# ── Engine ────────────────────────────────────────────────────────────────

def norm(v):
    return (v - 70) / 29

def circuit_weights(c):
    sr = c['straight_length_m'] / (c['lap_length_km'] * 1000)
    cd = c['corners'] / c['lap_length_km']
    w_spd, w_cor, w_brk = sr * 3, cd / 5, 0.30
    t = w_spd + w_cor + w_brk
    return w_spd/t, w_cor/t, w_brk/t

def perf_score_race(row, w_spd, w_cor, w_brk):
    bike = (
        w_spd * (norm(row['top_speed']) * 0.6 + norm(row['acceleration']) * 0.4)
        + w_cor * norm(row['bike_cornering'])
        + w_brk * norm(row['bike_braking'])
    )
    rw_cor = w_cor + w_spd * 0.3
    rw_brk = w_brk + w_spd * 0.5
    rw_agg = w_spd * 0.2
    rw_tot = rw_cor + rw_brk + rw_agg
    rider = (
        (rw_cor / rw_tot) * norm(row['rider_cornering'])
        + (rw_brk / rw_tot) * norm(row['rider_braking'])
        + (rw_agg / rw_tot) * norm(row['aggression'])
    )
    return 0.55 * bike + 0.45 * rider

def simulate_race_lap(row, lap_num, total_laps, score, is_wet):
    # ── Crash check (from lap 2) ──────────────────────────────────────────
    if lap_num > 1:
        crash_prob = 0.003 + 0.004 * norm(row['aggression']) * (1 - norm(row['consistency']))
        if np.random.random() < crash_prob:
            return None  # DNF

    # ── Base pace ────────────────────────────────────────────────────────
    time_pen = (1 - score) * MAX_DELTA

    # ── Tyre degradation (accelerates toward race end) ────────────────────
    tyre_deg = 2.0 * (1 - norm(row['tyre_management'])) * (lap_num / total_laps) ** 1.5

    # ── Fuel load (bike gets lighter → faster) ────────────────────────────
    fuel_gain = 0.4 * (lap_num - 1) / max(total_laps - 1, 1)

    # ── Standing start / first-lap traffic penalty ────────────────────────
    start_pen = {1: 4.0, 2: 1.5}.get(lap_num, 0.0)

    # ── Wet conditions ────────────────────────────────────────────────────
    wet_pen = 2.5 * (1 - norm(row['wet_performance'])) if is_wet else 0.0

    # ── Lap-to-lap variance (higher in lap 1-2) ───────────────────────────
    var_mult = 2.0 if lap_num <= 2 else 1.0
    variance = var_mult * 0.5 * (1 - norm(row['consistency'])) * (1 - norm(row['stability']))
    noise    = np.random.uniform(-variance, variance)

    lap_sec = base_time + time_pen + tyre_deg - fuel_gain + start_pen + wet_pen + noise
    return max(lap_sec, base_time * 0.97)

def fmt_lap(s):
    m = int(s // 60)
    return f'{m:02d}:{s % 60:06.3f}'

def fmt_gap(g):
    return '—' if g == 0 else f'+{g:.3f}'

def pos_arrow(prev, cur):
    d = prev - cur
    if d > 0: return f'▲{d}'
    if d < 0: return f'▼{-d}'
    return '='

print('Engine ready.')

Engine ready.


In [41]:
# ── Single-race runner ──────────────────────────────────────────────────────
# Scoring per point_scoring_system.md (MotoGP):
POINTS = {1: 25, 2: 20, 3: 16, 4: 13, 5: 11, 6: 10, 7: 9, 8: 8,
          9: 7, 10: 6, 11: 5, 12: 4, 13: 3, 14: 2, 15: 1}

w_spd, w_cor, w_brk = circuit_weights(circuit)
scores = {
    row['name']: perf_score_race(row, w_spd, w_cor, w_brk)
    for _, row in grid_df.iterrows()
}
title = f"{circuit['circuit_name']} - GRAND PRIX OF {country.upper()}"


def run_race(race_num):
    """Simulate one race, print + export its result, and return per-race rider points."""
    race_label = f'RACE {race_num}'

    # Weather rolls fresh for each race
    wet_threshold = np.random.uniform(0, 5)
    is_wet        = np.random.uniform(0, 100) <= wet_threshold
    weather_label = 'WET 🌧' if is_wet else 'DRY ☀'

    # ── Race state ──
    state = {}
    for _, rider in grid_df.iterrows():
        state[rider['name']] = {
            'bike_number'  : int(rider['bike_number']),
            'team'         : rider['team'],
            'manufacturer' : rider['manufacturer'],
            'cumul_time'   : 0.0,
            'position'     : int(rider['grid_pos']),
            'prev_position': int(rider['grid_pos']),
            'dnf'          : False,
            'dnf_lap'      : None,
            'lap_times'    : [],
        }
    dnf_log    = []
    overall_fl = {'time': float('inf'), 'name': None, 'lap': None}

    print('=' * 82)
    print(f'  {race_label}  —  {title}')
    print(f'  {total_laps} laps  |  {weather_label}')
    print('=' * 82)

    # ── Main loop (silent) ──
    for lap_num in range(1, total_laps + 1):
        lap_fastest = {'time': float('inf'), 'name': None}
        for name, s in state.items():
            if s['dnf']:
                continue
            rider_row = grid_df[grid_df['name'] == name].iloc[0]
            lap_sec   = simulate_race_lap(rider_row, lap_num, total_laps, scores[name], is_wet)
            if lap_sec is None:
                s['dnf']     = True
                s['dnf_lap'] = lap_num
                dnf_log.append({'lap': lap_num, 'name': name, 'bike_number': s['bike_number'],
                                'team': s['team'], 'manufacturer': s['manufacturer']})
            else:
                s['cumul_time'] += lap_sec
                s['lap_times'].append(lap_sec)
                if lap_sec < lap_fastest['time']:
                    lap_fastest = {'time': lap_sec, 'name': name}
        active = sorted([(n, s) for n, s in state.items() if not s['dnf']],
                        key=lambda x: x[1]['cumul_time'])
        for pos, (name, s) in enumerate(active, 1):
            s['prev_position'] = s['position']
            s['position']      = pos
        if lap_fastest['time'] < overall_fl['time']:
            overall_fl = {**lap_fastest, 'lap': lap_num}

    # ── Final classification ──
    finishers   = sorted([(n, s) for n, s in state.items() if not s['dnf']],
                         key=lambda x: x[1]['position'])
    dnf_sorted  = sorted(dnf_log, key=lambda x: -x['lap'])
    leader_time = finishers[0][1]['cumul_time'] if finishers else 0

    print(f"\n  {race_label} — FINAL CLASSIFICATION   ({weather_label})")
    print(f"  {'P':<4} {'#':<5} {'RIDER':<24} {'TEAM':<26} {'MANUFACTURER':<14} {'RACE TIME':>11} {'GAP':>9}")
    print(f"  {'─'*80}")
    for pos, (name, s) in enumerate(finishers, 1):
        gap    = fmt_gap(s['cumul_time'] - leader_time)
        fl_tag = '  ⚡' if name == overall_fl['name'] else ''
        print(f"  P{pos:<3} #{s['bike_number']:<4} {name:<24} {s['team']:<26} {s['manufacturer']:<14} {fmt_lap(s['cumul_time']):>11} {gap:>9}{fl_tag}")
    for d in dnf_sorted:
        print(f"  DNF  #{d['bike_number']:<4} {d['name']:<24} {d['team']:<26} {d['manufacturer']:<14} {'':>11} Lap {d['lap']}")
    print(f"  ⚡ Fastest lap: #{state[overall_fl['name']]['bike_number']} {overall_fl['name']} — {fmt_lap(overall_fl['time'])} (Lap {overall_fl['lap']})")

    # ── Export race{N}_result.md ──
    header  = '| P | # | RIDER | TEAM | MANUFACTURER | RACE TIME | GAP |'
    sep     = '|---|---|-------|------|--------------|-----------|-----|'
    md_rows = []
    for pos, (name, s) in enumerate(finishers, 1):
        gap = fmt_gap(s['cumul_time'] - leader_time)
        fl  = ' ⚡' if name == overall_fl['name'] else ''
        md_rows.append(f"| P{pos} | #{s['bike_number']} | {name}{fl} | {s['team']} | {s['manufacturer']} | {fmt_lap(s['cumul_time'])} | {gap} |")
    for d in dnf_sorted:
        md_rows.append(f"| DNF | #{d['bike_number']} | {d['name']} | {d['team']} | {d['manufacturer']} | — | Lap {d['lap']} |")
    race_md = (
        f'# {title}\n\n'
        f'## {race_label} - Final Classification\n\n'
        f'**Conditions:** {weather_label}  \n'
        f'**Fastest lap:** #{state[overall_fl["name"]]["bike_number"]} {overall_fl["name"]} — {fmt_lap(overall_fl["time"])} (Lap {overall_fl["lap"]})\n\n'
        f'{header}\n{sep}\n' + '\n'.join(md_rows) + '\n'
    )
    (report_dir / f'race{race_num}_result.md').write_text(race_md, encoding='utf-8')
    print(f"  Saved: race{race_num}_result.md")

    # ── Per-race rider points (returned for the overall ranking) ──
    rider_rows = []
    for pos, (name, s) in enumerate(finishers, 1):
        rider_rows.append({'name': name, 'bike_number': s['bike_number'], 'team': s['team'],
                           'manufacturer': s['manufacturer'], 'points': POINTS.get(pos, 0)})
    for d in dnf_sorted:
        rider_rows.append({'name': d['name'], 'bike_number': d['bike_number'], 'team': d['team'],
                           'manufacturer': d['manufacturer'], 'points': 0})
    return pd.DataFrame(rider_rows)


print('Runner ready.')

Runner ready.


In [42]:
# ── Run all races, then build ONE overall championship ranking ──────────────
all_rider_pts = []
for race_num in range(1, NUM_RACES + 1):
    all_rider_pts.append(run_race(race_num))
    print()

# Riders: total points across all races
combined     = pd.concat(all_rider_pts, ignore_index=True)
rider_total  = (combined.groupby(['name', 'bike_number', 'team', 'manufacturer'], as_index=False)['points']
                .sum().sort_values('points', ascending=False).reset_index(drop=True))

# Teams: sum of their riders' points across all races
team_total   = (combined.groupby('team', as_index=False)['points'].sum()
                .sort_values('points', ascending=False).reset_index(drop=True))

# Manufacturers: best-placed bike PER race, summed across races
manu_per_race = [r.groupby('manufacturer', as_index=False)['points'].max() for r in all_rider_pts]
manu_total    = (pd.concat(manu_per_race).groupby('manufacturer', as_index=False)['points'].sum()
                 .sort_values('points', ascending=False).reset_index(drop=True))

# ── Console display ─────────────────────────────────────────────────────────
print('=' * 60)
print(f'  OVERALL CHAMPIONSHIP — {title}')
print(f'  (after {NUM_RACES} races)')
print('=' * 60)

print('\n  RIDERS')
print(f"  {'P':<4} {'#':<5} {'RIDER':<24} {'PTS':>4}")
print(f"  {'─'*40}")
for i, r in rider_total.iterrows():
    print(f"  {i+1:<4} #{r['bike_number']:<4} {r['name']:<24} {r['points']:>4}")

print('\n  TEAMS')
print(f"  {'P':<4} {'TEAM':<26} {'PTS':>4}")
print(f"  {'─'*38}")
for i, r in team_total.iterrows():
    print(f"  {i+1:<4} {r['team']:<26} {r['points']:>4}")

print('\n  MANUFACTURERS')
print(f"  {'P':<4} {'MANUFACTURER':<16} {'PTS':>4}")
print(f"  {'─'*28}")
for i, r in manu_total.iterrows():
    print(f"  {i+1:<4} {r['manufacturer']:<16} {r['points']:>4}")
print('=' * 60)

# ── Export single ranking.md ────────────────────────────────────────────────
def _md_riders(dframe):
    head = '| P | # | RIDER | TEAM | MANUFACTURER | POINTS |'
    sep  = '|---|---|-------|------|--------------|--------|'
    rows = [f"| {i+1} | #{r['bike_number']} | {r['name']} | {r['team']} | {r['manufacturer']} | {r['points']} |"
            for i, r in dframe.iterrows()]
    return f"{head}\n{sep}\n" + "\n".join(rows)

def _md_simple(dframe, col, label):
    head = f"| P | {label} | POINTS |"
    sep  = "|---|" + "-" * (len(label) + 2) + "|--------|"
    rows = [f"| {i+1} | {r[col]} | {r['points']} |" for i, r in dframe.iterrows()]
    return f"{head}\n{sep}\n" + "\n".join(rows)

ranking_md = (
    f"# {title}\n\n"
    f"# Overall Championship Ranking (after {NUM_RACES} races)\n\n"
    f"## Riders' Championship\n\n{_md_riders(rider_total)}\n\n"
    f"---\n\n"
    f"## Teams' Championship\n\n{_md_simple(team_total, 'team', 'TEAM')}\n\n"
    f"---\n\n"
    f"## Manufacturers' Championship\n\n{_md_simple(manu_total, 'manufacturer', 'MANUFACTURER')}\n"
)
(report_dir / 'ranking.md').write_text(ranking_md, encoding='utf-8')
print(f"\nSaved: ranking.md  →  {report_dir}")

  RACE 1  —  Liam Henderson Circuit - GRAND PRIX OF AUSTRALIA
  23 laps  |  DRY ☀

  RACE 1 — FINAL CLASSIFICATION   (DRY ☀)
  P    #     RIDER                    TEAM                       MANUFACTURER     RACE TIME       GAP
  ────────────────────────────────────────────────────────────────────────────────
  P1   #31   Niklas Hoffmann          Suzuki Factory Racing      Suzuki           33:46.158         —  ⚡
  P2   #90   Nathan Stewart           Ducati Factory Racing      Ducati           33:50.116    +3.957
  P3   #44   Lorenzo Russo            Yamaha Factory Racing      Yamaha           33:51.444    +5.285
  P4   #20   Kaito Sato               Kawasaki Factory Racing    Kawasaki         33:56.575   +10.417
  P5   #26   Javier Ruiz              Razor Racing               Ducati           33:58.507   +12.348
  P6   #76   João Sousa               Suzuki Factory Racing      Suzuki           34:01.901   +15.742
  P7   #35   Manuel Navarro           Falcon Racing              BMW       